In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Depression Reddit: Text Classification

Classifies Reddit posts as depression (1) or not depression (0) using three algorithms:
**Naïve Bayes**, **Logistic Regression** and a **Neural Network**.
The text is used as provided, with no extra pre-processing.

## 1. Load the data
The dataset is stored as a zip file in the `COMP723Data` folder on Google Drive.
We list the folder, open the zip and read the CSV file inside it into a pandas DataFrame.

In [ ]:
import os
import zipfile
import pandas as pd

# Path to the zipped dataset on Google Drive
ZIP_PATH = '/content/drive/MyDrive/COMP723Data/Depression Reddit Dataset.zip'

print(os.listdir('/content/drive/MyDrive/COMP723Data'))

zf = zipfile.ZipFile(ZIP_PATH)
print(zf.namelist())

In [ ]:
# Read the (only) CSV file inside the zip
csv_name = [name for name in zf.namelist() if name.endswith('.csv')][0]
df = pd.read_csv(zf.open(csv_name))

print("Dataset shape:", df.shape)
display(df.head())

## 2. Explore the data
`clean_text` holds the Reddit post and `is_depression` is the label
(1 = depression, 0 = not depression). We check for missing values, empty posts,
duplicate posts and the class balance.

In [ ]:
print("Column information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

# Remove posts with no text, as they cannot be classified
df = df.dropna(subset=['clean_text'])
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)

print("\nShape after removing missing/empty posts:", df.shape)
print("Duplicate posts:", df.duplicated('clean_text').sum())

print("\nClass counts:")
print(df['is_depression'].value_counts())
print("\nClass proportions:")
print(df['is_depression'].value_counts(normalize=True).round(3))

# Post length in words, to describe the data
lengths = df['clean_text'].str.split().str.len()
print("\nWords per post: mean =", round(lengths.mean(), 1),
      "| median =", lengths.median(), "| max =", lengths.max())

## 3. Text and labels

In [ ]:
X = df['clean_text']
y = df['is_depression']

print("Text:")
print(X.head())
print("\nLabels:")
print(y.head())

## 4. Stratified 80/20 train/test split
The dataset has no separate test set, so we split it 80/20, keeping the class
proportions the same in both sets (`stratify=y`). `random_state=42` gives the same
split every time, and the same split is used for all three algorithms.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining class counts:")
print(y_train.value_counts())
print("\nTesting class counts:")
print(y_test.value_counts())

## 5. TF-IDF vectorisation
Each post is turned into a vector of TF-IDF scores for the 5,000 most frequent words.
The vectoriser is fitted on the training set only, so no information from the test
set leaks into the features.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

## 6. Evaluation helper
Precision, recall and F1 are reported for the chosen category, **depression (label 1)**.
Every run is stored in `all_runs` together with its training time, so the summary
tables at the end are built from the actual results rather than typed in by hand.

In [ ]:
import time
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

all_runs = []

def record(algorithm, setting, predictions, train_time):
    """Print and store precision, recall and F1 for the depression class."""
    run = {
        'Algorithm': algorithm,
        'Setting': setting,
        'Precision': round(precision_score(y_test, predictions), 3),
        'Recall': round(recall_score(y_test, predictions), 3),
        'F1 Score': round(f1_score(y_test, predictions), 3),
        'Train time (s)': round(train_time, 2)
    }
    all_runs.append(run)
    print(setting, "| Precision:", run['Precision'], "| Recall:", run['Recall'],
          "| F1:", run['F1 Score'], "| Time:", run['Train time (s)'], "s")

## 7. Algorithm 1: Naïve Bayes
Multinomial Naïve Bayes with default settings first, then tuned over the smoothing
parameter `alpha` (0.1, 0.5, 1.0).

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Baseline with default settings (alpha = 1.0)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_predictions = nb_model.predict(X_test_tfidf)
print(classification_report(y_test, nb_predictions))

# Tuning: try different smoothing values
for alpha in [0.1, 0.5, 1.0]:
    start = time.time()
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_tfidf, y_train)
    train_time = time.time() - start

    predictions = model.predict(X_test_tfidf)
    record('Naive Bayes', f'alpha = {alpha}', predictions, train_time)

## 8. Algorithm 2 (our choice): Logistic Regression
Logistic Regression with default settings first, then tuned over `C`
(0.1, 1, 10). `C` is the inverse of the regularisation strength: a larger `C`
lets the model rely more heavily on individual words.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Baseline with default settings (C = 1)
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_predictions = lr_model.predict(X_test_tfidf)
print(classification_report(y_test, lr_predictions))

# Tuning: try different C values
for C in [0.1, 1, 10]:
    start = time.time()
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train_tfidf, y_train)
    train_time = time.time() - start

    predictions = model.predict(X_test_tfidf)
    record('Logistic Regression', f'C = {C}', predictions, train_time)

## 9. Algorithm 3: Neural Network
A feed-forward network with one hidden ReLU layer and a sigmoid output, trained with
Adam and binary cross-entropy for 5 epochs (20% of the training set is used for
validation). The baseline has 64 hidden units; we then try 32, 64 and 128.
The random seed is fixed so the results can be reproduced.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

def build_nn(neurons):
    """One hidden layer network for binary classification."""
    tf.keras.utils.set_random_seed(42)
    model = Sequential([
        Input(shape=(X_train_dense.shape[1],)),
        Dense(neurons, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Baseline with 64 hidden units (training history shown per epoch)
nn_model = build_nn(64)
nn_model.summary()
history = nn_model.fit(X_train_dense, y_train, epochs=5, batch_size=32,
                       validation_split=0.2)

nn_predictions = (nn_model.predict(X_test_dense) > 0.5).astype(int).flatten()
print(classification_report(y_test, nn_predictions))

In [ ]:
# Tuning: try different numbers of hidden units
for neurons in [32, 64, 128]:
    model = build_nn(neurons)

    start = time.time()
    model.fit(X_train_dense, y_train, epochs=5, batch_size=32,
              validation_split=0.2, verbose=0)
    train_time = time.time() - start

    predictions = (model.predict(X_test_dense, verbose=0) > 0.5).astype(int).flatten()
    record('Neural Network', f'{neurons} neurons', predictions, train_time)

## 10. Results
All tuning runs, followed by the best setting (highest F1) for each algorithm.

In [ ]:
runs_df = pd.DataFrame(all_runs)
print("All runs:")
display(runs_df)

best_results = runs_df.loc[runs_df.groupby('Algorithm')['F1 Score'].idxmax()].reset_index(drop=True)
print("Best setting per algorithm:")
display(best_results)